# Preparazione dei dataframe per la classificazione
Per alleggerire i notebook sulla classificazione, prepariamo separatamente i dataframe. Si organizza questo notebook mantenendo al minimo i commenti. Si contestualizzano quindi i dataframe nel modo in cui vengono utilizzati.

In [1]:
# importiamo i pacchetti necessari
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from  sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn import metrics

import sys
sys.path.append('../..')

import src.class_funcs as fs

In [2]:
# importiamo il dataframe sviluppato per l'EDA
dataset_df = pd.read_csv('../../data/processed/dataset_EDA_processed.csv')

# inseriamo in dataset_df una colonna 'day' che contiene il giorno del mese estratto dalla colonna 'date'
dataset_df['date'] = pd.to_datetime(dataset_df['date'])
dataset_df['day'] = (dataset_df["date"] - dataset_df["date"].min()).dt.days + 1

dataset_df.head()

,station_appa,date,hour,CO,NO2,O3,PM10,PM2.5,SO2,geometry_appa,...,temperature,precipitation,winds_spd,winds_dir,geometry_weather,power_area_50,power_area_1000,power_area_2500,week_day,day
0,Borgo Valsugana,2013-11-01,1,NaN,21.0,2.0,19.0,12.0,NaN,POINT (11.45389 46.05184),...,11.350,0.0,NaN,NaN,POINT (11.47747769 46.05804607),17.760273,42.946772,54.704690,friday,1
1,Borgo Valsugana,2013-11-01,2,NaN,18.0,2.0,18.0,11.0,NaN,POINT (11.45389 46.05184),...,11.525,0.0,NaN,NaN,POINT (11.47747769 46.05804607),16.362439,39.762416,52.679229,friday,1
2,Borgo Valsugana,2013-11-01,3,NaN,19.0,2.0,19.0,12.0,NaN,POINT (11.45389 46.05184),...,11.425,0.0,NaN,NaN,POINT (11.47747769 46.05804607),17.861264,41.283915,50.294214,friday,1
3,Borgo Valsugana,2013-11-01,4,NaN,17.0,2.0,20.0,11.0,NaN,POINT (11.45389 46.05184),...,11.075,0.0,NaN,NaN,POINT (11.47747769 46.05804607),14.669913,36.119279,45.105859,friday,1
4,Borgo Valsugana,2013-11-01,5,NaN,18.0,2.0,21.0,13.0,NaN,POINT (11.45389 46.05184),...,10.950,0.0,NaN,NaN,POINT (11.47747769 46.05804607),16.969367,39.626135,48.951222,friday,1


## Classificazione binaria
per la classificazione binaria vogliamo usare le seguenti features:
- 'station_appa' in formato one-hot: stazione di rilevazione;

- 'elevation': elevazione delle stazioni;

- 'day': processione ordinata di giorni dal primo dato;

- 'cos_week_day': coseno di periodicità del giorno della settimana;

- 'sin_week_day': seno di periodicità del giorno della settimana;

- 'cos_hour': coseno di periodicità dell'ora;

- 'sin_hour': seno di periodicità dell'ora;

- 'PM10_1', 'NO2_1', 'AQI_1': inquinanti e indice un'ora prima;

- 'PM10_2', 'NO2_2', 'AQI_2': inquinanti e indice due ore prima;

- 'PM10_3', 'NO2_3', 'AQI_3': inquinanti e indice tre ore prima;

- 'power_area_50_1', 'power_area_50_2', 'power_area_50_3': consumi elettrici entro 50 metri per una, due e tre ore prima;

- 'PM10_diff', 'NO2_diff', 'AQI_diff': variazione di inquinanti e indice tra una e due ore prima;

- 'temperature': temperatura un'ora prima;

- 'winds_spd': velocità del vento un'ora prima;

- 'precipitation': precipitazioni di 5 ore prima (nell'EDA sembravano quelle correlate maggiormente).

Come 'target' mettiamo 1 = 'ok' se 'EAQI' è 'good' o 'fair' e 0 = 'not ok' se 'EAQI' è 'moderate', 'poor' o 'very poor'

NB: 'winds_spd' è molto correlato con gli inquinanti quindi sarebbe in via di principio una feature rilevante. Purtroppo la maggior parte delle rilevazioni del vento sono assenti e la colonna è pertanto piena di NaN. Poiché i modelli non possono lavorare con NaN, siamo obbligati a rinunciare alla colonna o a riempire i vuoti. Come visto nel notebook di appendice "09_winds_appendix.ipynb", la scelta migliore è rinunciare al dato quindi eliminiamo la colonna 'winds_spd' come prima operazione in ciascun notebook.

In [3]:
# feature istantanee
binary_class_df = dataset_df[['station_appa', 'elevation', 'day', 'week_day', 'hour']].copy()
binary_class_df = pd.get_dummies(binary_class_df, columns=['station_appa'], prefix='station', dtype=int) # stazioni in one-hot

## scriviamoil giorno della settimana e l'ora come seni e coseni
week_day_map = {'monday':0, 'tuesday':1, 'wednesday':2, 'thursday':3, 'friday':4, 'saturday':5, 'sunday':6}
binary_class_df['week_day'] = binary_class_df['week_day'].map(week_day_map) # ritrasformo i giorni della settimana in numeri da 0 a 6

### giorni della settimana
binary_class_df['cos_week_day'] = np.cos(binary_class_df['week_day'] * 2*np.pi/7)
binary_class_df['sin_week_day'] = np.sin(binary_class_df['week_day'] * 2*np.pi/7)

### ore
binary_class_df['cos_hour'] = np.cos(binary_class_df['hour'] * 2*np.pi/24)
binary_class_df['sin_hour'] = np.sin(binary_class_df['hour'] * 2*np.pi/24)

## droppiamo le colonne vecchie
binary_class_df.drop(columns=['hour', 'week_day'], inplace=True)

# riordiniamo le colonne
station_cols = [col for col in binary_class_df.columns if col.startswith('station_')]
binary_class_df = binary_class_df[station_cols + ['elevation', 'day', 'cos_week_day', 'sin_week_day', 'cos_hour', 'sin_hour']]

# aggiungiamo le colonne con gli inquinanti ed i consumi elettrici spostati di 1, 2 e 3 ore
pol_cols = ['PM10', 'NO2', 'AQI']
shift_h = [1, 2, 3]
for h in shift_h:
    binary_class_df[[col + f'_{h}' for col in pol_cols]] = dataset_df.groupby('station_appa')[pol_cols].shift(h)
    binary_class_df[f'power_area_50_{h}'] = dataset_df.groupby('station_appa')['power_area_50'].shift(h)

# aggiungiamo le colonne con il meteo
binary_class_df['temperature'] = dataset_df.groupby('station_appa')['temperature'].shift(1)
binary_class_df['winds_spd'] = dataset_df.groupby('station_appa')['winds_spd'].shift(1)
binary_class_df['precipitation'] = dataset_df.groupby('station_appa')['precipitation'].shift(5)

# aggiungiamo le colonne con la variazione di inquinanti ed indice
binary_class_df[[col + '_diff' for col in pol_cols]] = dataset_df.groupby('station_appa')[pol_cols].diff(1).shift(1)

# aggiungiamo il target
binary_class_df['target'] = ((dataset_df['EAQI'] == 'good') | (dataset_df['EAQI'] == 'fair')).astype(int)

# eliminiamo le righe contenenti NaN in colonne che non siano 'winds_spd'
non_winds = binary_class_df.columns.difference(['winds_spd'])
binary_class_df.dropna(subset=non_winds, inplace=True)

# salviamo il dataframe
binary_class_df.to_csv("../../data/processed/dataset_binary_class_processed.csv", index=False)

binary_class_df.head()

,station_Borgo Valsugana,station_Monte Gaza,station_Parco S. Chiara,station_Piana Rotaliana,station_Riva del Garda,station_Rovereto,station_Via Bolzano,elevation,day,cos_week_day,...,NO2_3,AQI_3,power_area_50_3,temperature,winds_spd,precipitation,PM10_diff,NO2_diff,AQI_diff,target
5,1,0,0,0,0,0,0,410,1,-0.900969,...,19.0,24.0,17.861264,10.950,NaN,0.0,1.0,1.0,4.0,1
6,1,0,0,0,0,0,0,410,1,-0.900969,...,17.0,22.0,14.669913,11.000,NaN,0.0,-2.0,-2.0,0.0,1
7,1,0,0,0,0,0,0,410,1,-0.900969,...,18.0,26.0,16.969367,10.925,NaN,0.0,5.0,0.0,6.0,1
8,1,0,0,0,0,0,0,410,1,-0.900969,...,16.0,26.0,15.329278,10.950,NaN,0.0,-5.0,-1.0,-10.0,1
9,1,0,0,0,0,0,0,410,1,-0.900969,...,16.0,32.0,17.519438,11.550,NaN,0.0,3.0,0.0,2.0,1


## Classificazione a 5 classi
Per la classificazione a 5 classi usiamo le stesse feature che per la classificazione binaria.

Come target usiamo le 5 classi 'good', 'fair', 'moderate', 'poor' e 'very poor'.

In [4]:
# costruiamo il dataframe come fatto per quello binario

# feature istantanee
multiclass_df = dataset_df[['station_appa', 'elevation', 'day', 'week_day', 'hour']].copy()
multiclass_df = pd.get_dummies(multiclass_df, columns=['station_appa'], prefix='station', dtype=int) # stazioni in one-hot

## scriviamoil giorno della settimana e l'ora come seni e coseni
week_day_map = {'monday':0, 'tuesday':1, 'wednesday':2, 'thursday':3, 'friday':4, 'saturday':5, 'sunday':6}
multiclass_df['week_day'] = multiclass_df['week_day'].map(week_day_map) # ritrasformo i giorni della settimana in numeri da 0 a 6

### giorni della settimana
multiclass_df['cos_week_day'] = np.cos(multiclass_df['week_day'] * 2*np.pi/7)
multiclass_df['sin_week_day'] = np.sin(multiclass_df['week_day'] * 2*np.pi/7)

### ore
multiclass_df['cos_hour'] = np.cos(multiclass_df['hour'] * 2*np.pi/24)
multiclass_df['sin_hour'] = np.sin(multiclass_df['hour'] * 2*np.pi/24)

## droppiamo le colonne vecchie
multiclass_df.drop(columns=['hour', 'week_day'], inplace=True)

# riordiniamo le colonne
station_cols = [col for col in multiclass_df.columns if col.startswith('station_')]
multiclass_df = multiclass_df[station_cols + ['elevation', 'day', 'cos_week_day', 'sin_week_day', 'cos_hour', 'sin_hour']]

# aggiungiamo le colonne con gli inquinanti ed i consumi elettrici spostati di 1, 2 e 3 ore
pol_cols = ['PM10', 'NO2', 'AQI']
shift_h = [1, 2, 3]
for h in shift_h:
    multiclass_df[[col + f'_{h}' for col in pol_cols]] = dataset_df.groupby('station_appa')[pol_cols].shift(h)
    multiclass_df[f'power_area_50_{h}'] = dataset_df.groupby('station_appa')['power_area_50'].shift(h)

# aggiungiamo le colonne con il meteo
multiclass_df['temperature'] = dataset_df.groupby('station_appa')['temperature'].shift(1)
multiclass_df['winds_spd'] = dataset_df.groupby('station_appa')['winds_spd'].shift(1)
multiclass_df['precipitation'] = dataset_df.groupby('station_appa')['precipitation'].shift(5)

# aggiungiamo le colonne con la variazione di inquinanti ed indice
multiclass_df[[col + '_diff' for col in pol_cols]] = dataset_df.groupby('station_appa')[pol_cols].diff(1).shift(1)

# aggiungiamo il target
multiclass_df['target'] = dataset_df['EAQI']

# eliminiamo le righe contenenti NaN in colonne che non siano 'winds_spd'
non_winds = multiclass_df.columns.difference(['winds_spd'])
multiclass_df.dropna(subset=non_winds, inplace=True)

# salviamo il dataframe
multiclass_df.to_csv("../../data/processed/dataset_multiclass_processed.csv", index=False)

multiclass_df.head()

,station_Borgo Valsugana,station_Monte Gaza,station_Parco S. Chiara,station_Piana Rotaliana,station_Riva del Garda,station_Rovereto,station_Via Bolzano,elevation,day,cos_week_day,...,NO2_3,AQI_3,power_area_50_3,temperature,winds_spd,precipitation,PM10_diff,NO2_diff,AQI_diff,target
5,1,0,0,0,0,0,0,410,1,-0.900969,...,19.0,24.0,17.861264,10.950,NaN,0.0,1.0,1.0,4.0,fair
6,1,0,0,0,0,0,0,410,1,-0.900969,...,17.0,22.0,14.669913,11.000,NaN,0.0,-2.0,-2.0,0.0,fair
7,1,0,0,0,0,0,0,410,1,-0.900969,...,18.0,26.0,16.969367,10.925,NaN,0.0,5.0,0.0,6.0,fair
8,1,0,0,0,0,0,0,410,1,-0.900969,...,16.0,26.0,15.329278,10.950,NaN,0.0,-5.0,-1.0,-10.0,fair
9,1,0,0,0,0,0,0,410,1,-0.900969,...,16.0,32.0,17.519438,11.550,NaN,0.0,3.0,0.0,2.0,fair


## Classificazione giornaliera
per la classificazione giornaliera vogliamo usare le seguenti features:
- 'station_appa' in formato one-hot: stazione di rilevazione;

- 'elevation': elevazione delle stazioni;

- 'day': processione ordinata di giorni dal primo dato;

- 'cos_week_day': coseno di periodicità del giorno della settimana;

- 'sin_week_day': seno di periodicità del giorno della settimana;

- 'PM10_1', 'NO2_1', 'AQI_1': inquinanti e indice medi del giorno prima;

- 'PM10_2', 'NO2_2', 'AQI_2': inquinanti e indice medi di due giorni prima;

- 'power_area_50_1', 'power_area_50_2': consumi elettrici medi entro 50 metri di uno e due giorni prima;

- 'PM10_diff', 'NO2_diff', 'AQI_diff': variazione di inquinanti e indice tra uno e due giorni prima;

- 'PM10_std', 'NO2_std', 'AQI_std': deviazione standard di inquinanti e indice del giorno prima;

- 'temperature': temperatura media del giorno prima;

- 'precipitation': precipitazioni medie del giorno prima.

Come 'target' mettiamo 1 = 'ok' se 'EAQI' è 'good' o 'fair' e 0 = 'not ok' se 'EAQI' è 'moderate', 'poor' o 'very poor'

NB: togliamo i venti, avendo capito di non riuscirli ad integrare in maniera rilevante.

In [5]:
# costruiamo il dataframe accorpando i dati delle ore diverse
temp_df = (dataset_df.groupby(['station_appa', 'day'], as_index=False).agg(
    elevation=('elevation', 'first'),
    week_day=('week_day', 'first'),
    PM10_mean=('PM10', 'mean'),
    NO2_mean=('NO2', 'mean'),
    AQI_mean=('AQI', 'mean'),
    PM10_std=('PM10', 'std'),
    NO2_std=('NO2', 'std'),
    AQI_std=('AQI', 'std'),
    power_area_50=('power_area_50', 'mean'),
    temperature=('temperature', 'mean'),
    precipitation=('precipitation', 'mean')
))

# non specifichiamo le ore, in questo modo vengono abbandonate poiché sono inutili a questo studio
# inoltre abbandoniamo l'EAQI poiché non numerico e comunque non lo useremmo per il training dei modelli
# abbandoniamo anche winds_spd visto che è composto principalmente da NaN

# scriviamo il giorno della settimana come seno e coseno
week_day_map = {'monday':0, 'tuesday':1, 'wednesday':2, 'thursday':3, 'friday':4, 'saturday':5, 'sunday':6}
temp_df['week_day'] = temp_df['week_day'].map(week_day_map) # ritrasformo i giorni della settimana in numeri da 0 a 6

temp_df['cos_week_day'] = np.cos(temp_df['week_day'] * 2*np.pi/7)
temp_df['sin_week_day'] = np.sin(temp_df['week_day'] * 2*np.pi/7)

# aggiungiamo le feature del giorno stesso
daily_df = temp_df['station_appa'].copy()

## scriviamo le stazioni in formato one-hot
daily_df = pd.get_dummies(daily_df, columns=['station_appa'], prefix='station', dtype=int)
daily_df[['elevation', 'day', 'cos_week_day', 'sin_week_day']] = temp_df[['elevation', 'day', 'cos_week_day', 'sin_week_day']]

# aggiungiamo le colonne con gli inquinanti ed i consumi elettrici spostati di 1 e 2 giorni
mean_pol_cols = ['PM10_mean', 'NO2_mean', 'AQI_mean']

shift_d = [1, 2]
for d in shift_d:
    daily_df[[col + f'_{d}' for col in mean_pol_cols]] = temp_df.groupby('station_appa')[mean_pol_cols].shift(d)
    daily_df[f'power_area_50_{d}'] = temp_df.groupby('station_appa')['power_area_50'].shift(d)

# aggiungiamo le colonne con la variazione di inquinanti ed indice medi tra uno e due giorni prima
daily_df[[col + '_diff' for col in mean_pol_cols]] = temp_df.groupby('station_appa')[mean_pol_cols].diff(1).shift(1)

# aggiungiamo le colonne con la deviazione standard degli inquinanti per il giorno prima
std_pol_cols = ['PM10_std', 'NO2_std', 'AQI_std']

daily_df[std_pol_cols] = temp_df.groupby('station_appa')[std_pol_cols].shift(1)

# aggiungiamo le colonne con il meteo
daily_df['temperature'] = temp_df.groupby('station_appa')['temperature'].shift(1)
daily_df['precipitation'] = temp_df.groupby('station_appa')['precipitation'].shift(1)

# aggiungiamo il target
daily_df['target'] = (temp_df['AQI_mean'] <= 40).astype(int)

# eliminiamo le righe che abbiamo appena aggiunto con valori NaN
daily_df.dropna(inplace=True)
daily_df = daily_df.reset_index(drop=True)

# salviamo il dataframe
daily_df.to_csv("../../data/processed/dataset_daily_class_processed.csv", index=False)

daily_df.head()

,station_Borgo Valsugana,station_Monte Gaza,station_Parco S. Chiara,station_Piana Rotaliana,station_Riva del Garda,station_Rovereto,station_Via Bolzano,elevation,day,cos_week_day,...,power_area_50_2,PM10_mean_diff,NO2_mean_diff,AQI_mean_diff,PM10_std,NO2_std,AQI_std,temperature,precipitation,target
0,1,0,0,0,0,0,0,410,3,0.623490,...,17.843233,3.130435,2.173913,14.394203,6.717036,8.564796,16.634925,11.231522,0.013043,1
1,1,0,0,0,0,0,0,410,4,1.000000,...,18.197335,-5.217391,-2.260870,-10.298551,9.585067,6.004939,20.707330,10.796739,0.126087,1
2,1,0,0,0,0,0,0,410,5,0.623490,...,17.478475,-8.913043,7.913043,-13.305797,9.036378,7.995552,14.470189,7.609783,0.028261,1
3,1,0,0,0,0,0,0,410,6,-0.222521,...,37.200400,2.260870,-0.826087,4.273913,10.527022,9.380410,16.751478,9.641304,0.000000,1
4,1,0,0,0,0,0,0,410,7,-0.900969,...,37.421266,-2.260870,1.043478,-5.864493,5.310516,10.332993,6.320607,9.980435,0.000000,1
